In [0]:
import os

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_path = os.path.dirname(notebook_path)
os.environ["PROJECTCWD"] = project_path


%pip install --quiet --upgrade pystac pystac-client planetary_computer tenacity rich osdatahub

In [0]:
# restart python
dbutils.library.restartPython()

In [0]:
# verify download from azure blob

import pandas as pd
import numpy as np
import planetary_computer

In [0]:
pdf_geo_sample = pd.read_csv("/Workspace/Users/liang.cheng@augustschell.com/sentinel2-eo-golf-aws/sample_azure_geospatial_href.csv")
display(pdf_geo_sample)

In [0]:
import requests
import os

url = "https://planetarycomputer.microsoft.com/api/sas/v1/token/sentinel2l2a01/sentinel2-l2"
response = requests.get(url)
response_data = response.json()
display(response_data)

# set environment variable
os.environ["PC_SDK_SUBSCRIPTION_KEY"] = response_data["token"]

In [0]:
def download_asset(href, dir_path):
  import requests
  import os.path
  from tenacity import retry, wait_exponential, TryAgain, stop_after_attempt, RetryError

  filename = os.path.basename(href.split("?")[0])
  outpath = f"{dir_path}/{filename}"
  if os.path.exists(outpath):
    return outpath
  
  @retry(
    wait=wait_exponential(multiplier=2, min=4, max=120),
    stop=stop_after_attempt(5)
    )
  def retryable(href, dir_path, filename):
    try:
      print(f"Downloading {filename} from {href} to {outpath}")
      # Make the actual request, set the timeout for no data to 10 seconds and enable streaming responses so we don't have to keep the large files in memory
      href = href.split("?")[0]
      signed_url = planetary_computer.sign(href)
      response = requests.get(signed_url, timeout=100, stream=True)
      if int(response.status_code) != 200 or int(response.headers['content-length']) < 1024:
        print(f"Downloading {filename} from {href} failed. Response {response} Trying again.")
        raise TryAgain
      # Open the output file and make sure we write in binary mode
      with open(outpath, 'wb') as fh:
        # Walk through the request response in chunks of 1024 * 1024 bytes, so 1MiB
        for chunk in response.iter_content(1024 * 1024):
          # Write the chunk to the file
          fh.write(chunk)
          # Optionally we can check here if the download is taking too long
      return outpath
    except RetryError as e:
      return f"Error downloading {filename}: {e}"
    except:
      raise TryAgain
  return retryable(href, dir_path, filename)

In [0]:
# iterate through each row of pdf_geo_sample

for index, row in pdf_geo_sample.iterrows():
    href = row['href']
    dir_path = '/Workspace/Users/liang.cheng@augustschell.com/sentinel2-eo-golf-aws'
    download_asset(href, dir_path)